The car routes include congested times.  For each TBI record, find the corresponding time, and match to the appropriate routes for that time. 

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd

import keyring

In [2]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

In [3]:
# This is used to avoid hard-coding directories. 
# To set the directory, use the command prompt or a notebook you don't check in.  run:
# import keyring
# keyring.set_password("msp", "vmt_reduction_dir", <directory>)

# get base path for data
data_dir = keyring.get_password("msp", "vmt_reduction_dir")

In [4]:
def get_car_congestion_period(row):
    hour = int(row["arrive_time"][0:2])
    sunday = row["travel_dow"] == "Sunday"
    saturday = row["travel_dow"] == "Saturday"
    if sunday:
        query = "sundays_"
    elif saturday:
        query = "saturdays_"
    else:
        query = "weekdays_"
    
    if hour >= 0 and hour <= 5:
        query += "0-6"
    elif hour >= 20 and hour <= 23:
        query += "20-24"
    else:
        query += str(hour) + "-" + str(hour + 1)

    return query

In [5]:
# read in the data
tbi = pd.read_csv(data_dir + "/data_processed/tbi_cleaned.csv")

C:\Users\ger225\AppData\Local\Temp\ipykernel_26728\902543372.py:2: DtypeWarning: Columns (33,34,35,52,64,65,66,67,68,69,70) have mixed types. Specify dtype option on import or set low_memory=False.
  tbi = pd.read_csv(data_dir + "/data_processed/tbi_cleaned.csv")


In [6]:
tbi['congestion_period'] = tbi.apply(get_car_congestion_period, axis=1)

In [7]:
# select the appropriate rows from the congested car data
periods = tbi['congestion_period'].unique()

for period in periods:
    selected_trips = tbi[tbi['congestion_period']==period]    
    selected_trip_ids = selected_trips['trip_id']
    
    print ('Processing ' + period + ' with ' + str(len(selected_trip_ids)) + ' trips.')
    
    all_routes = gpd.read_parquet(data_dir + "/Data_Processed/geodata/car_scenario/car_scenario_" + period + '-5d901.parquet')
    
    selected_routes = all_routes.merge(selected_trip_ids, on='trip_id', how='inner')
    selected_routes.to_parquet(data_dir + "/Data_processed/geodata/car_scenario/selected_car_scenario_" + period + ".parquet", index=False)

Processing weekdays_13-14 with 20186 trips.
Processing weekdays_20-24 with 50363 trips.
Processing weekdays_6-7 with 14917 trips.
Processing weekdays_15-16 with 27456 trips.
Processing weekdays_11-12 with 16251 trips.
Processing weekdays_14-15 with 23368 trips.
Processing weekdays_17-18 with 26584 trips.
Processing weekdays_18-19 with 20779 trips.
Processing weekdays_16-17 with 30731 trips.
Processing weekdays_10-11 with 13137 trips.
Processing weekdays_12-13 with 17163 trips.
Processing weekdays_7-8 with 16366 trips.
Processing weekdays_9-10 with 10699 trips.
Processing sundays_16-17 with 4028 trips.
Processing sundays_17-18 with 3607 trips.
Processing weekdays_0-6 with 27705 trips.
Processing weekdays_19-20 with 16444 trips.
Processing weekdays_8-9 with 12327 trips.
Processing saturdays_10-11 with 3733 trips.
Processing saturdays_11-12 with 3702 trips.
Processing saturdays_12-13 with 3743 trips.
Processing saturdays_13-14 with 3845 trips.
Processing saturdays_14-15 with 4260 trips.
P

In [9]:
# now merge the files
periods = tbi['congestion_period'].unique()

gdf = gpd.GeoDataFrame()
for period in periods:
    gdf_period = gpd.read_parquet(data_dir + "/Data_processed/geodata/car_scenario/selected_car_scenario_" + period + ".parquet")
    gdf = pd.concat([gdf, gdf_period])
    
gdf.to_parquet(data_dir + "/Data_processed/geodata/car_scenario.parquet", index=False)